# NanoFlux on Colab
### Flux.1 Dev (generate) + Flux.1 Kontext Dev (edit) in one ComfyUI workflow

This notebook installs ComfyUI, downloads the fp8-quantized Flux.1 Dev and
Flux.1 Kontext Dev checkpoints (sized for a single L4 GPU), drops in a
ready-made workflow with two switchable groups, and gives you a public URL
to open ComfyUI in your browser.

**Before running:** you need a Hugging Face account and an access token, and
you must accept the license on both of these pages (click "Agree" while
logged in) — the downloads below will fail with a 401 error otherwise:
- https://huggingface.co/black-forest-labs/FLUX.1-dev
- https://huggingface.co/black-forest-labs/FLUX.1-Kontext-dev

Get a token (read access is enough) at https://huggingface.co/settings/tokens

**Runtime:** Runtime -> Change runtime type -> select the **L4 GPU**, then run every cell top to bottom.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Hugging Face token
Saves the token as an environment variable. If you've added `HF_TOKEN` to
Colab's Secrets panel (key icon in the left sidebar) it's picked up
automatically; otherwise you'll be prompted to paste it.

In [ ]:
import os
from getpass import getpass

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    pass

if not hf_token:
    hf_token = getpass("Paste your Hugging Face token (https://huggingface.co/settings/tokens): ")

os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token
print("HF token set:", bool(hf_token))

## 3. Install ComfyUI

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI
!pip install -q -r requirements.txt
!pip install -q huggingface_hub

# ComfyUI-Manager -- lets you fix/install any missing custom nodes from inside the UI
!git clone --depth 1 https://github.com/Comfy-Org/ComfyUI-Manager.git custom_nodes/ComfyUI-Manager

for d in ["models/diffusion_models", "models/text_encoders", "models/vae", "user/default/workflows"]:
    os.makedirs(f"/content/ComfyUI/{d}", exist_ok=True)
print("ComfyUI installed.")

## 4. Download the models
~29 GB total (fp8-scaled Dev + fp8-scaled Kontext + text encoders + VAE) --
sized to run comfortably on the L4's 24 GB. Takes a few minutes on Colab's
connection. If any file 401s, you haven't accepted that model's license yet
(see the links in step 0).

In [ ]:
from huggingface_hub import hf_hub_download
import shutil

MODELS_DIR = "/content/ComfyUI/models"

def fetch(repo_id, filename, dest_subdir, dest_filename=None):
    dest_filename = dest_filename or filename.split("/")[-1]
    dest_path = f"{MODELS_DIR}/{dest_subdir}/{dest_filename}"
    if os.path.exists(dest_path):
        print(f"already have {dest_path}")
        return
    print(f"downloading {repo_id}/{filename} -> {dest_path}")
    try:
        cached = hf_hub_download(repo_id=repo_id, filename=filename, token=hf_token)
    except Exception as e:
        print(f"  FAILED: {e}")
        print(f"  -> make sure you've accepted the license for {repo_id} on huggingface.co")
        return
    shutil.copyfile(cached, dest_path)
    print(f"  done ({os.path.getsize(dest_path) / 1e9:.2f} GB)")

# Text encoders + VAE (ungated)
fetch("comfyanonymous/flux_text_encoders", "clip_l.safetensors", "text_encoders")
fetch("comfyanonymous/flux_text_encoders", "t5xxl_fp8_e4m3fn.safetensors", "text_encoders")
fetch("black-forest-labs/FLUX.1-schnell", "ae.safetensors", "vae")

# Diffusion models (gated -- license must be accepted first)
fetch("comfyanonymous/flux_dev_scaled_fp8_test", "flux_dev_fp8_scaled_diffusion_model.safetensors",
      "diffusion_models", dest_filename="flux1-dev.safetensors")
fetch("Comfy-Org/flux1-kontext-dev_ComfyUI", "split_files/diffusion_models/flux1-dev-kontext_fp8_scaled.safetensors",
      "diffusion_models")

print("\nAll downloads attempted. Check above for any FAILED lines before continuing.")

## 5. Install the NanoFlux workflow
Writes the two-group (Generate / Edit) workflow into ComfyUI's workflow
folder so it's already loaded when the UI opens.

In [ ]:
nanoflux_workflow = '{\n  "last_node_id": 26,\n  "last_link_id": 27,\n  "nodes": [\n    {\n      "id": 1,\n      "type": "DualCLIPLoader",\n      "pos": [\n        420,\n        -420\n      ],\n      "size": [\n        340,\n        130\n      ],\n      "flags": {},\n      "order": 0,\n      "mode": 0,\n      "inputs": [],\n      "outputs": [\n        {\n          "name": "CLIP",\n          "type": "CLIP",\n          "links": [\n            1,\n            2\n          ],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "DualCLIPLoader"\n      },\n      "widgets_values": [\n        "t5xxl_fp8_e4m3fn.safetensors",\n        "clip_l.safetensors",\n        "flux",\n        "default"\n      ]\n    },\n    {\n      "id": 2,\n      "type": "VAELoader",\n      "pos": [\n        420,\n        -250\n      ],\n      "size": [\n        340,\n        80\n      ],\n      "flags": {},\n      "order": 1,\n      "mode": 0,\n      "inputs": [],\n      "outputs": [\n        {\n          "name": "VAE",\n          "type": "VAE",\n          "links": [\n            3,\n            4,\n            5\n          ],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "VAELoader"\n      },\n      "widgets_values": [\n        "ae.safetensors"\n      ]\n    },\n    {\n      "id": 3,\n      "type": "Note",\n      "pos": [\n        420,\n        -600\n      ],\n      "size": [\n        500,\n        140\n      ],\n      "flags": {},\n      "order": 2,\n      "mode": 0,\n      "inputs": [],\n      "outputs": [],\n      "properties": {\n        "Node name for S&R": "Note"\n      },\n      "widgets_values": [\n        "NANOFLUX \\u2014 Flux.1 Dev + Kontext Dev hybrid workflow\\n\\nGroup \\u2460 (left) = GENERATE a brand new image from text.\\nGroup \\u2461 (right) = EDIT the last image (drag its thumbnail onto the Load Image node).\\n\\nOnly ONE group should be un-muted at a time. See the notes in each group."\n      ]\n    },\n    {\n      "id": 4,\n      "type": "UNETLoader",\n      "pos": [\n        0,\n        -420\n      ],\n      "size": [\n        340,\n        110\n      ],\n      "flags": {},\n      "order": 3,\n      "mode": 0,\n      "inputs": [],\n      "outputs": [\n        {\n          "name": "MODEL",\n          "type": "MODEL",\n          "links": [\n            6\n          ],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "UNETLoader"\n      },\n      "widgets_values": [\n        "flux1-dev.safetensors",\n        "default"\n      ],\n      "title": "Load Diffusion Model (Flux.1 Dev)"\n    },\n    {\n      "id": 5,\n      "type": "ModelSamplingFlux",\n      "pos": [\n        0,\n        -260\n      ],\n      "size": [\n        340,\n        140\n      ],\n      "flags": {},\n      "order": 4,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "model",\n          "type": "MODEL",\n          "link": 6\n        }\n      ],\n      "outputs": [\n        {\n          "name": "MODEL",\n          "type": "MODEL",\n          "links": [\n            7\n          ],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "ModelSamplingFlux"\n      },\n      "widgets_values": [\n        1.15,\n        0.5,\n        1024,\n        1024\n      ]\n    },\n    {\n      "id": 6,\n      "type": "CLIPTextEncode",\n      "pos": [\n        0,\n        -90\n      ],\n      "size": [\n        340,\n        200\n      ],\n      "flags": {},\n      "order": 5,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "clip",\n          "type": "CLIP",\n          "link": 1\n        }\n      ],\n      "outputs": [\n        {\n          "name": "CONDITIONING",\n          "type": "CONDITIONING",\n          "links": [\n            8,\n            9\n          ],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "CLIPTextEncode"\n      },\n      "widgets_values": [\n        "a photo of ..."\n      ],\n      "title": "Prompt (what to generate)"\n    },\n    {\n      "id": 7,\n      "type": "FluxGuidance",\n      "pos": [\n        0,\n        140\n      ],\n      "size": [\n        340,\n        80\n      ],\n      "flags": {},\n      "order": 6,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "conditioning",\n          "type": "CONDITIONING",\n          "link": 8\n        }\n      ],\n      "outputs": [\n        {\n          "name": "CONDITIONING",\n          "type": "CONDITIONING",\n          "links": [\n            10\n          ],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "FluxGuidance"\n      },\n      "widgets_values": [\n        3.5\n      ]\n    },\n    {\n      "id": 8,\n      "type": "ConditioningZeroOut",\n      "pos": [\n        0,\n        250\n      ],\n      "size": [\n        340,\n        60\n      ],\n      "flags": {},\n      "order": 7,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "conditioning",\n          "type": "CONDITIONING",\n          "link": 9\n        }\n      ],\n      "outputs": [\n        {\n          "name": "CONDITIONING",\n          "type": "CONDITIONING",\n          "links": [\n            11\n          ],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "ConditioningZeroOut"\n      },\n      "widgets_values": []\n    },\n    {\n      "id": 9,\n      "type": "EmptySD3LatentImage",\n      "pos": [\n        0,\n        340\n      ],\n      "size": [\n        340,\n        130\n      ],\n      "flags": {},\n      "order": 8,\n      "mode": 0,\n      "inputs": [],\n      "outputs": [\n        {\n          "name": "LATENT",\n          "type": "LATENT",\n          "links": [\n            12\n          ],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "EmptySD3LatentImage"\n      },\n      "widgets_values": [\n        1024,\n        1024,\n        1\n      ]\n    },\n    {\n      "id": 10,\n      "type": "KSampler",\n      "pos": [\n        380,\n        0\n      ],\n      "size": [\n        340,\n        300\n      ],\n      "flags": {},\n      "order": 9,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "model",\n          "type": "MODEL",\n          "link": 7\n        },\n        {\n          "name": "positive",\n          "type": "CONDITIONING",\n          "link": 10\n        },\n        {\n          "name": "negative",\n          "type": "CONDITIONING",\n          "link": 11\n        },\n        {\n          "name": "latent_image",\n          "type": "LATENT",\n          "link": 12\n        }\n      ],\n      "outputs": [\n        {\n          "name": "LATENT",\n          "type": "LATENT",\n          "links": [\n            13\n          ],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "KSampler"\n      },\n      "widgets_values": [\n        0,\n        "randomize",\n        20,\n        1.0,\n        "euler",\n        "simple",\n        1.0\n      ]\n    },\n    {\n      "id": 11,\n      "type": "VAEDecode",\n      "pos": [\n        380,\n        340\n      ],\n      "size": [\n        340,\n        60\n      ],\n      "flags": {},\n      "order": 10,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "samples",\n          "type": "LATENT",\n          "link": 13\n        },\n        {\n          "name": "vae",\n          "type": "VAE",\n          "link": 3\n        }\n      ],\n      "outputs": [\n        {\n          "name": "IMAGE",\n          "type": "IMAGE",\n          "links": [\n            14\n          ],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "VAEDecode"\n      },\n      "widgets_values": []\n    },\n    {\n      "id": 12,\n      "type": "SaveImage",\n      "pos": [\n        380,\n        440\n      ],\n      "size": [\n        340,\n        260\n      ],\n      "flags": {},\n      "order": 11,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "images",\n          "type": "IMAGE",\n          "link": 14\n        }\n      ],\n      "outputs": [],\n      "properties": {\n        "Node name for S&R": "SaveImage"\n      },\n      "widgets_values": [\n        "nanoflux/generate"\n      ]\n    },\n    {\n      "id": 13,\n      "type": "Note",\n      "pos": [\n        0,\n        720\n      ],\n      "size": [\n        340,\n        160\n      ],\n      "flags": {},\n      "order": 12,\n      "mode": 0,\n      "inputs": [],\n      "outputs": [],\n      "properties": {\n        "Node name for S&R": "Note"\n      },\n      "widgets_values": [\n        "\\u2460 GENERATE\\n\\nWrite a prompt above, make sure THIS group is un-muted, and mute Group \\u2461 before queuing.\\nRight-click group title bar -> Set Group Nodes to Never to mute."\n      ]\n    },\n    {\n      "id": 14,\n      "type": "UNETLoader",\n      "pos": [\n        900,\n        -420\n      ],\n      "size": [\n        340,\n        110\n      ],\n      "flags": {},\n      "order": 13,\n      "mode": 2,\n      "inputs": [],\n      "outputs": [\n        {\n          "name": "MODEL",\n          "type": "MODEL",\n          "links": [\n            15\n          ],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "UNETLoader"\n      },\n      "widgets_values": [\n        "flux1-dev-kontext_fp8_scaled.safetensors",\n        "default"\n      ],\n      "title": "Load Diffusion Model (Flux.1 Kontext Dev)"\n    },\n    {\n      "id": 15,\n      "type": "ModelSamplingFlux",\n      "pos": [\n        900,\n        -260\n      ],\n      "size": [\n        340,\n        140\n      ],\n      "flags": {},\n      "order": 14,\n      "mode": 2,\n      "inputs": [\n        {\n          "name": "model",\n          "type": "MODEL",\n          "link": 15\n        }\n      ],\n      "outputs": [\n        {\n          "name": "MODEL",\n          "type": "MODEL",\n          "links": [\n            16\n          ],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "ModelSamplingFlux"\n      },\n      "widgets_values": [\n        1.15,\n        0.5,\n        1024,\n        1024\n      ]\n    },\n    {\n      "id": 16,\n      "type": "LoadImage",\n      "pos": [\n        900,\n        -90\n      ],\n      "size": [\n        340,\n        320\n      ],\n      "flags": {},\n      "order": 15,\n      "mode": 2,\n      "inputs": [],\n      "outputs": [\n        {\n          "name": "IMAGE",\n          "type": "IMAGE",\n          "links": [\n            17\n          ],\n          "slot_index": 0\n        },\n        {\n          "name": "MASK",\n          "type": "MASK",\n          "links": [],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "LoadImage"\n      },\n      "widgets_values": [\n        "put_your_image_here.png",\n        "image"\n      ],\n      "title": "Load Image (drag last generated/edited image here)"\n    },\n    {\n      "id": 17,\n      "type": "FluxKontextImageScale",\n      "pos": [\n        900,\n        270\n      ],\n      "size": [\n        340,\n        60\n      ],\n      "flags": {},\n      "order": 16,\n      "mode": 2,\n      "inputs": [\n        {\n          "name": "image",\n          "type": "IMAGE",\n          "link": 17\n        }\n      ],\n      "outputs": [\n        {\n          "name": "IMAGE",\n          "type": "IMAGE",\n          "links": [\n            18\n          ],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "FluxKontextImageScale"\n      },\n      "widgets_values": []\n    },\n    {\n      "id": 18,\n      "type": "VAEEncode",\n      "pos": [\n        900,\n        370\n      ],\n      "size": [\n        340,\n        70\n      ],\n      "flags": {},\n      "order": 17,\n      "mode": 2,\n      "inputs": [\n        {\n          "name": "pixels",\n          "type": "IMAGE",\n          "link": 18\n        },\n        {\n          "name": "vae",\n          "type": "VAE",\n          "link": 4\n        }\n      ],\n      "outputs": [\n        {\n          "name": "LATENT",\n          "type": "LATENT",\n          "links": [\n            22,\n            25\n          ],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "VAEEncode"\n      },\n      "widgets_values": []\n    },\n    {\n      "id": 19,\n      "type": "CLIPTextEncode",\n      "pos": [\n        1280,\n        -90\n      ],\n      "size": [\n        340,\n        200\n      ],\n      "flags": {},\n      "order": 18,\n      "mode": 2,\n      "inputs": [\n        {\n          "name": "clip",\n          "type": "CLIP",\n          "link": 2\n        }\n      ],\n      "outputs": [\n        {\n          "name": "CONDITIONING",\n          "type": "CONDITIONING",\n          "links": [\n            19,\n            20\n          ],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "CLIPTextEncode"\n      },\n      "widgets_values": [\n        "change ..."\n      ],\n      "title": "Edit instruction (what to change)"\n    },\n    {\n      "id": 20,\n      "type": "FluxGuidance",\n      "pos": [\n        1280,\n        140\n      ],\n      "size": [\n        340,\n        80\n      ],\n      "flags": {},\n      "order": 19,\n      "mode": 2,\n      "inputs": [\n        {\n          "name": "conditioning",\n          "type": "CONDITIONING",\n          "link": 19\n        }\n      ],\n      "outputs": [\n        {\n          "name": "CONDITIONING",\n          "type": "CONDITIONING",\n          "links": [\n            21\n          ],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "FluxGuidance"\n      },\n      "widgets_values": [\n        2.5\n      ]\n    },\n    {\n      "id": 21,\n      "type": "ConditioningZeroOut",\n      "pos": [\n        1280,\n        250\n      ],\n      "size": [\n        340,\n        60\n      ],\n      "flags": {},\n      "order": 20,\n      "mode": 2,\n      "inputs": [\n        {\n          "name": "conditioning",\n          "type": "CONDITIONING",\n          "link": 20\n        }\n      ],\n      "outputs": [\n        {\n          "name": "CONDITIONING",\n          "type": "CONDITIONING",\n          "links": [\n            24\n          ],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "ConditioningZeroOut"\n      },\n      "widgets_values": []\n    },\n    {\n      "id": 22,\n      "type": "ReferenceLatent",\n      "pos": [\n        900,\n        470\n      ],\n      "size": [\n        340,\n        70\n      ],\n      "flags": {},\n      "order": 21,\n      "mode": 2,\n      "inputs": [\n        {\n          "name": "conditioning",\n          "type": "CONDITIONING",\n          "link": 21\n        },\n        {\n          "name": "latent",\n          "type": "LATENT",\n          "link": 22\n        }\n      ],\n      "outputs": [\n        {\n          "name": "CONDITIONING",\n          "type": "CONDITIONING",\n          "links": [\n            23\n          ],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "ReferenceLatent"\n      },\n      "widgets_values": []\n    },\n    {\n      "id": 23,\n      "type": "KSampler",\n      "pos": [\n        1280,\n        340\n      ],\n      "size": [\n        340,\n        300\n      ],\n      "flags": {},\n      "order": 22,\n      "mode": 2,\n      "inputs": [\n        {\n          "name": "model",\n          "type": "MODEL",\n          "link": 16\n        },\n        {\n          "name": "positive",\n          "type": "CONDITIONING",\n          "link": 23\n        },\n        {\n          "name": "negative",\n          "type": "CONDITIONING",\n          "link": 24\n        },\n        {\n          "name": "latent_image",\n          "type": "LATENT",\n          "link": 25\n        }\n      ],\n      "outputs": [\n        {\n          "name": "LATENT",\n          "type": "LATENT",\n          "links": [\n            26\n          ],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "KSampler"\n      },\n      "widgets_values": [\n        0,\n        "randomize",\n        20,\n        1.0,\n        "euler",\n        "simple",\n        1.0\n      ]\n    },\n    {\n      "id": 24,\n      "type": "VAEDecode",\n      "pos": [\n        1280,\n        680\n      ],\n      "size": [\n        340,\n        60\n      ],\n      "flags": {},\n      "order": 23,\n      "mode": 2,\n      "inputs": [\n        {\n          "name": "samples",\n          "type": "LATENT",\n          "link": 26\n        },\n        {\n          "name": "vae",\n          "type": "VAE",\n          "link": 5\n        }\n      ],\n      "outputs": [\n        {\n          "name": "IMAGE",\n          "type": "IMAGE",\n          "links": [\n            27\n          ],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "VAEDecode"\n      },\n      "widgets_values": []\n    },\n    {\n      "id": 25,\n      "type": "SaveImage",\n      "pos": [\n        1280,\n        780\n      ],\n      "size": [\n        340,\n        260\n      ],\n      "flags": {},\n      "order": 24,\n      "mode": 2,\n      "inputs": [\n        {\n          "name": "images",\n          "type": "IMAGE",\n          "link": 27\n        }\n      ],\n      "outputs": [],\n      "properties": {\n        "Node name for S&R": "SaveImage"\n      },\n      "widgets_values": [\n        "nanoflux/edit"\n      ]\n    },\n    {\n      "id": 26,\n      "type": "Note",\n      "pos": [\n        900,\n        1080\n      ],\n      "size": [\n        720,\n        170\n      ],\n      "flags": {},\n      "order": 25,\n      "mode": 0,\n      "inputs": [],\n      "outputs": [],\n      "properties": {\n        "Node name for S&R": "Note"\n      },\n      "widgets_values": [\n        "\\u2461 EDIT\\n\\nDrag the thumbnail of the image you want to edit onto the Load Image node above (or click it to upload one).\\nWrite the change you want in the text box, un-mute THIS group, and mute Group \\u2460 before queuing.\\nRight-click group title bar -> Set Group Nodes to Always to un-mute."\n      ]\n    }\n  ],\n  "links": [\n    [\n      1,\n      1,\n      0,\n      6,\n      0,\n      "CLIP"\n    ],\n    [\n      2,\n      1,\n      0,\n      19,\n      0,\n      "CLIP"\n    ],\n    [\n      3,\n      2,\n      0,\n      11,\n      1,\n      "VAE"\n    ],\n    [\n      4,\n      2,\n      0,\n      18,\n      1,\n      "VAE"\n    ],\n    [\n      5,\n      2,\n      0,\n      24,\n      1,\n      "VAE"\n    ],\n    [\n      6,\n      4,\n      0,\n      5,\n      0,\n      "MODEL"\n    ],\n    [\n      7,\n      5,\n      0,\n      10,\n      0,\n      "MODEL"\n    ],\n    [\n      8,\n      6,\n      0,\n      7,\n      0,\n      "CONDITIONING"\n    ],\n    [\n      9,\n      6,\n      0,\n      8,\n      0,\n      "CONDITIONING"\n    ],\n    [\n      10,\n      7,\n      0,\n      10,\n      1,\n      "CONDITIONING"\n    ],\n    [\n      11,\n      8,\n      0,\n      10,\n      2,\n      "CONDITIONING"\n    ],\n    [\n      12,\n      9,\n      0,\n      10,\n      3,\n      "LATENT"\n    ],\n    [\n      13,\n      10,\n      0,\n      11,\n      0,\n      "LATENT"\n    ],\n    [\n      14,\n      11,\n      0,\n      12,\n      0,\n      "IMAGE"\n    ],\n    [\n      15,\n      14,\n      0,\n      15,\n      0,\n      "MODEL"\n    ],\n    [\n      16,\n      15,\n      0,\n      23,\n      0,\n      "MODEL"\n    ],\n    [\n      17,\n      16,\n      0,\n      17,\n      0,\n      "IMAGE"\n    ],\n    [\n      18,\n      17,\n      0,\n      18,\n      0,\n      "IMAGE"\n    ],\n    [\n      19,\n      19,\n      0,\n      20,\n      0,\n      "CONDITIONING"\n    ],\n    [\n      20,\n      19,\n      0,\n      21,\n      0,\n      "CONDITIONING"\n    ],\n    [\n      21,\n      20,\n      0,\n      22,\n      0,\n      "CONDITIONING"\n    ],\n    [\n      22,\n      18,\n      0,\n      22,\n      1,\n      "LATENT"\n    ],\n    [\n      23,\n      22,\n      0,\n      23,\n      1,\n      "CONDITIONING"\n    ],\n    [\n      24,\n      21,\n      0,\n      23,\n      2,\n      "CONDITIONING"\n    ],\n    [\n      25,\n      18,\n      0,\n      23,\n      3,\n      "LATENT"\n    ],\n    [\n      26,\n      23,\n      0,\n      24,\n      0,\n      "LATENT"\n    ],\n    [\n      27,\n      24,\n      0,\n      25,\n      0,\n      "IMAGE"\n    ]\n  ],\n  "groups": [\n    {\n      "title": "\\u2460 GENERATE \\u2014 new image from text (Flux.1 Dev)",\n      "bounding": [\n        -40,\n        -480,\n        800,\n        1000\n      ],\n      "color": "#3f789e",\n      "font_size": 24,\n      "locked": false\n    },\n    {\n      "title": "\\u2461 EDIT \\u2014 change the last image (Flux.1 Kontext Dev)",\n      "bounding": [\n        860,\n        -480,\n        800,\n        1770\n      ],\n      "color": "#a1309b",\n      "font_size": 24,\n      "locked": false\n    }\n  ],\n  "config": {},\n  "extra": {\n    "ds": {\n      "scale": 0.6,\n      "offset": [\n        200,\n        600\n      ]\n    }\n  },\n  "version": 0.4\n}'

with open("/content/ComfyUI/user/default/workflows/nanoflux.json", "w") as f:
    f.write(nanoflux_workflow)

print("Workflow installed.")

## 6. Launch ComfyUI + get your public URL
Starts ComfyUI in the background, then opens a Cloudflare tunnel and prints
the `https://...trycloudflare.com` link -- open that link to use ComfyUI.
Leave this cell running; stopping it kills the tunnel.

In [ ]:
import subprocess, time, re, threading

# Install cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

%cd /content/ComfyUI

# Start ComfyUI in the background
comfy_log = open("/content/comfyui.log", "w")
comfy_proc = subprocess.Popen(
    ["python", "main.py", "--listen", "0.0.0.0", "--port", "8188"],
    stdout=comfy_log, stderr=subprocess.STDOUT
)
print("Starting ComfyUI (this can take ~30-60s the first time)...")
time.sleep(20)

# Open the tunnel and grab the public URL
cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

url_found = threading.Event()

def watch():
    for line in cf_proc.stdout:
        m = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", line)
        if m and not url_found.is_set():
            print("\n" + "=" * 60)
            print("ComfyUI is live at:", m.group(0))
            print("(open the NanoFlux workflow from the Workflows menu if it isn't already loaded)")
            print("=" * 60 + "\n")
            url_found.set()

threading.Thread(target=watch, daemon=True).start()

for _ in range(60):
    if url_found.is_set():
        break
    time.sleep(1)

if not url_found.is_set():
    print("Tunnel URL not detected yet -- check /content/comfyui.log for errors, or re-run this cell.")
